# ATML PA3 — Colab bootstrap

Run this notebook top-to-bottom on **Colab Pro (L4, 22 GB)** at the start of every session.
It mounts Drive, points `HF_HOME` and `DRIVE_ROOT` at Drive (so caches/checkpoints survive
disconnects), pulls the repo, installs requirements, **reminds you to restart the runtime**,
then validates the environment (GPU, versions, Qwen3 tokenizer + non-thinking chat template).

Edit `REPO_URL` and `DRIVE_ROOT` below if needed.

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Set DRIVE_ROOT + HF_HOME on Drive and create the layout.
import os
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/ATML-PA3'   # approved layout
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'  # model/tokenizer cache survives disconnects

subdirs = [
    'data', 'hf_cache',
    'task1/checkpoints', 'task1/eval', 'task1/generations',
    'task2/features', 'task2/checkpoints', 'task2/eval',
    'task3/live', 'task3/cache',
]
for d in subdirs:
    Path(f'{DRIVE_ROOT}/{d}').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('HF_HOME    =', os.environ['HF_HOME'])
print('Put supplied files in', f'{DRIVE_ROOT}/data', ':',
      'first_plot_questions.yaml (Task 1), ATML-PA3-Data/ (Task 3)')

In [ ]:
# 3) Clone or pull the repo.
REPO_URL = 'https://github.com/<your-user>/ATML-PA3.git'  # TODO: set to your repo
REPO_DIR = '/content/ATML-PA3'
import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR

In [ ]:
# 4) Install requirements (uses Colab's preinstalled torch).
!pip install -q -r requirements.txt

## ⚠️ 5) Restart the runtime now

`pip install` may have upgraded `transformers` / other libs. **Restart the runtime**
(Runtime ▸ Restart session), then **re-run cells 1–3** (mount + env + `%cd`) and continue from
the validation cell below. Do **not** re-run cell 4 after restarting.

In [ ]:
# 6) Validate the environment (run AFTER the restart + re-running cells 1-3).
import os, torch, transformers
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
assert os.environ.get('DRIVE_ROOT'), 'DRIVE_ROOT not set — re-run cell 2'
assert os.environ.get('HF_HOME'),    'HF_HOME not set — re-run cell 2'
# transformers >= 4.51 expected for Qwen3 support (verify the exact working version here).
print('HF_HOME      :', os.environ['HF_HOME'])

In [ ]:
# 7) Confirm the Qwen3 tokenizer loads and the non-thinking chat template works.
from transformers import AutoTokenizer, AutoConfig
name = 'Qwen/Qwen3-0.6B'
tok = AutoTokenizer.from_pretrained(name)
cfg = AutoConfig.from_pretrained(name)
print('hidden_size      :', getattr(cfg, 'hidden_size', None), '(configs assume 1024)')
print('num_hidden_layers:', getattr(cfg, 'num_hidden_layers', None), '(diagnostics assume 28 -> layers 0,13,27)')
msgs = [{'role': 'user', 'content': 'What is 2+2?'}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print('--- chat template (enable_thinking=False) ---')
print(prompt)
assert '<think>' not in prompt, 'Thinking block present — enable_thinking=False not honored?'